# Basic example

### 4 hours of prices, using only basic battery info

In [1]:
hourly_prices = [6,3,2,5] #units £/MWh

In [2]:
max_charging_rate = 2 #units MW
max_discharging_rate = 2 #units MW
max_storage_volume = 4 #units MWh

In [3]:
charging_efficiency = 0.05 #Fraction of energy imported from grid that is lost prior to storage in the battery
discharging_efficiency = 0.05 #Fraction of energy exported from the battery that is lost prior to reaching the grid

### Aim: maxmise profit, can decide when to charge and discharge, can't decide what prices are

In [4]:
import pulp

In [5]:
model = pulp.LpProblem("battery_optimisation", pulp.LpMaximize)

In [6]:
n_hours = len(hourly_prices)
print(n_hours)

4


In [7]:
# Define variables at each time step, state of charge (soc) is useful to link charge and discharge
charge = pulp.LpVariable.dicts("charge", range(n_hours), 0, max_charging_rate)
discharge = pulp.LpVariable.dicts("discharge", range(n_hours), 0, max_discharging_rate)
soc = pulp.LpVariable.dicts("soc", range(n_hours), 0, max_storage_volume)

In [8]:
print(discharge[2])

discharge_2


In [9]:
# profit = sum( price * discharge * efficiency - price * charge )

model += pulp.lpSum(
    hourly_prices[t] * discharge[t] * (1 - discharging_efficiency) - hourly_prices[t] * charge[t]
    for t in range(n_hours)
)

In [10]:
# state of charge = state of charge at previous hour + charge * efficiency - discharge  --- this links charge and discharge together, essentially conservation of energy

initial_soc = 0.0

for t in range(n_hours):
    previous_soc = initial_soc if t == 0 else soc[t - 1]
    model += soc[t] == previous_soc + charge[t] * (1 - charging_efficiency) - discharge[t]

In [11]:
model.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/constancemahony/miniconda3/envs/pulp-env/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/55/qvzldl952577jnfzcs8thwqc0000gn/T/19a35f0ad4384bd8838220446fecb91d-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/55/qvzldl952577jnfzcs8thwqc0000gn/T/19a35f0ad4384bd8838220446fecb91d-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 33 RHS
At line 38 BOUNDS
At line 51 ENDATA
Problem MODEL has 4 rows, 12 columns and 15 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 3 (-1) rows, 9 (-3) columns and 11 (-4) elements
0  Obj -0 Dual inf 9.499997 (3)
3  Obj 5.1842105
Optimal - objective value 5.1842105
After Postsolve, objective 5.1842105, infeasibilities - dual 2.5421052 (1), primal 0 (0)
Presolved model was optimal, full model needs c

1

In [12]:
print(f"Status: {pulp.LpStatus[model.status]}\n")
for t in range(n_hours):
    print(f"Hour {t}: price=£{hourly_prices[t]:3d}  "
          f"charge={charge[t].varValue:.2f}  "
          f"discharge={discharge[t].varValue:.2f}  "
          f"SoC={soc[t].varValue:.2f}")

profit = pulp.value(model.objective)
print(f"\nTotal profit: £{profit:.2f}")

Status: Optimal

Hour 0: price=£  6  charge=0.00  discharge=0.00  SoC=-0.00
Hour 1: price=£  3  charge=0.11  discharge=0.00  SoC=0.10
Hour 2: price=£  2  charge=2.00  discharge=0.00  SoC=2.00
Hour 3: price=£  5  charge=0.00  discharge=2.00  SoC=0.00

Total profit: £5.18


# write function

In [ ]:
def battery_optimiser(prices, max_charging_rate, max_discharging_rate, max_storage_volume, charging_efficiency )

# Now read in data

In [13]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'